# Training and Evaluating xRFM

In [ ]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler

from xrfm import xRFM
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier



c:\Users\jbrag\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading and pre-processing
Data is split into training, validation, and test sets and each is transformed with normal scaler. Non continuous data is represented as one-hot-encoded values.

In [11]:

mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(np.int64)
X = X / 255.0


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)


Train size: (56000, 784)
Validation size: (7000, 784)
Test size: (7000, 784)


## Instatiating models and setting hyper parameters

xRFM model and two benchmark models are instantiated and hyper-parameters set. Different versions and tuning metrics are used for Regression and Classification tasks. 

In [12]:
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model_xgb = XGBClassifier(
    n_estimators=100,
    # max_depth=8,
    # learning_rate=0.1,
    objective="multi:softprob",
    num_class=10,
    tree_method="hist",
    max_bin=64,
    n_jobs=-1,
    random_state=42
)

rfm_params = {
    'model': {
        'kernel': 'l2',           # Kernel type
        'bandwidth': 5,         # Kernel bandwidth
        'exponent': 1,          # Kernel exponent
        'diag': True,            # Diagonal Mahalanobis matrix
        'bandwidth_mode': 'constant'
    },
    'fit': {
        'reg': 1,              # Regularization parameter
        'iters': 0,               # Number of iterations
        'verbose': False,          # Verbose output
        'early_stop_rfm': False    # Early stopping
    }
}

model_xrfm = xRFM(
    rfm_params=rfm_params,
    tuning_metric='auc',
)


None


## Training models

In [13]:
# RF
start = time.time()
model_rf.fit(X_train_scaled, y_train)
end = time.time()
rf_train_time = end - start

print(f"Random Forest training time: {(rf_train_time):.4f} seconds")

# xgBoost
start = time.time()
model_xgb.fit(X_train_scaled, y_train)
end = time.time()
xgb_train_time = end - start
print(f"XGBoost training time: {(xgb_train_time):.4f} seconds")

# xRFM
start = time.time()
model_xrfm.fit(X_train_scaled, y_train, X_val_scaled, y_val)
end = time.time()
xrfm_train_time = end - start
print(f"XRFM training time: {(xrfm_train_time):.4f} seconds")


NameError: name 'time' is not defined

## Evaluation and results

In [ ]:
# RF
start = time.time()
y_pred_rf = model_rf.predict(X_test_scaled)
end = time.time()
rf_pred_time = end - start

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(f"AUC-ROC score")
print(roc_auc_score(y_test, model_rf.predict_proba(X_test_scaled), multi_class="ovr"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

# xgBoost
start = time.time()
y_pred_xgb = model_xgb.predict(X_test_scaled)
end = time.time()
xgb_pred_time = end - start
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(f"AUC-ROC score")
print(roc_auc_score(y_test, model_xgb.predict_proba(X_test_scaled), multi_class="ovr"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xgb))

# xRFM
start = time.time()
y_pred_xrfm = model_xrfm.predict(X_val_scaled)
end = time.time()
xrfm_pred_time = end - start
print("XRFM Accuracy:", accuracy_score(y_val, y_pred_xrfm))
print(f"AUC-ROC score")
print(roc_auc_score(y_test, model_xrfm.predict_proba(X_test_scaled), multi_class="ovr"))
print("\nClassification Report:\n")
print(classification_report(y_val, y_pred_xrfm))